# Four-class UNSW-NB15 experiment

This is an independent experiment. It keeps Normal, Exploits, Fuzzers, and DoS. Analysis and Backdoor are excluded from this experiment only. The existing fit and validation row indices are reused after filtering, so no new random split is made.

In [ ]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

EXPERIMENT = Path('.').resolve()
PROJECT = EXPERIMENT.parent.parent
DATA = PROJECT / 'data' / 'processed'
RESULTS = EXPERIMENT / 'results'
PLOTS = RESULTS / 'plots'
MODELS = EXPERIMENT / 'models'
SEED = 42
THREADS = 2
CLASSES = ['Normal', 'Exploits', 'Fuzzers', 'DoS']   # four classes 
assert not any((RESULTS / name).exists() for name in ['model_comparison.csv', 'per_class_metrics.csv', 'four_vs_six_comparison.csv'])
frame = pd.read_csv(DATA / 'selected_training.csv')    # need to check wheather we discarding the rows or not 
fit_rows = pd.read_csv(DATA / 'fit_indices.csv')['row_index'].to_numpy()
validation_rows = pd.read_csv(DATA / 'validation_indices.csv')['row_index'].to_numpy()
assert len(set(fit_rows) & set(validation_rows)) == 0
assert set(fit_rows) | set(validation_rows) == set(range(len(frame)))
predictors = [c for c in frame.columns if c not in {'id', 'label', 'attack_cat', '_fingerprint'}]
assert 'id' not in predictors and 'label' not in predictors and 'attack_cat' not in predictors
four_mask = frame['attack_cat'].isin(CLASSES)
fit_rows = fit_rows[np.isin(fit_rows, np.flatnonzero(four_mask))]
validation_rows = validation_rows[np.isin(validation_rows, np.flatnonzero(four_mask))]
fit_data = frame.iloc[fit_rows].copy()
validation_data = frame.iloc[validation_rows].copy()
X_fit = fit_data[predictors]
X_validation = validation_data[predictors]
y_fit = fit_data['attack_cat']
y_validation = validation_data['attack_cat']
assert set(y_fit) == set(CLASSES) and set(y_validation) == set(CLASSES)
fingerprints = pd.util.hash_pandas_object(frame[predictors], index=False, categorize=True)
fit_groups = set(fingerprints.iloc[fit_rows])
validation_groups = set(fingerprints.iloc[validation_rows])
assert not fit_groups.intersection(validation_groups)
numeric_features = X_fit.select_dtypes(include='number').columns.tolist()
categorical_features = X_fit.select_dtypes(exclude='number').columns.tolist()
print('Fit rows:', len(fit_data))
print('Validation rows:', len(validation_data))
print('Predictor count:', len(predictors))
print('Fit class counts:\n', y_fit.value_counts().reindex(CLASSES))
print('Validation class counts:\n', y_validation.value_counts().reindex(CLASSES))
print('Fingerprint overlap:', len(fit_groups.intersection(validation_groups)))

## Preprocessing

Numeric columns use median imputation. Categorical columns use the most common value and one-hot encoding. These steps are inside each pipeline, so they learn values from the fit partition only.

In [ ]:
def make_preprocessor(dense=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    categorical_steps = [('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=not dense, dtype=np.float32))]
    return ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), numeric_features),
        ('categorical', Pipeline(categorical_steps), categorical_features),
    ], sparse_threshold=0 if dense else 1)

def show_confusion(name, matrix):
    normalized = matrix / matrix.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for axis, values, title, number_format in zip(axes, [matrix, normalized], ['Raw', 'Normalized'], ['d', '.2f']):
        axis.imshow(values, cmap='Blues')
        axis.set(xticks=range(4), yticks=range(4), xticklabels=CLASSES, yticklabels=CLASSES, xlabel='Predicted', ylabel='Actual', title=f'{name} {title}')
        plt.setp(axis.get_xticklabels(), rotation=30, ha='right')
        for i in range(4):
            for j in range(4):
                axis.text(j, i, format(values[i, j], number_format), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(PLOTS / f'{name.lower().replace(" ", "_")}_confusion.png', dpi=130)
    plt.close(fig)

model_rows = []
class_rows = []
models = {}
def evaluate_model(name, pipeline, config, sample_weight=None, phase='baseline'):
    start = time.perf_counter()
    if sample_weight is None:
        pipeline.fit(X_fit, y_fit)
    else:
        pipeline.fit(X_fit, y_fit, classifier__sample_weight=sample_weight)
    fit_seconds = time.perf_counter() - start
    prediction = pipeline.predict(X_validation)
    report = classification_report(y_validation, prediction, labels=CLASSES, output_dict=True, zero_division=0)
    matrix = confusion_matrix(y_validation, prediction, labels=CLASSES)
    normal_rows = y_validation.to_numpy() == 'Normal'
    attack_rows = ~normal_rows
    attack_prediction = prediction != 'Normal'
    row = {'model': name, 'phase': phase, 'configuration': json.dumps(config, sort_keys=True, default=str), 'fit_seconds': fit_seconds, 'accuracy': accuracy_score(y_validation, prediction), 'macro_f1': f1_score(y_validation, prediction, average='macro'), 'weighted_f1': f1_score(y_validation, prediction, average='weighted'), 'normal_false_positive_rate': float(attack_prediction[normal_rows].mean()), 'attack_recall': float(attack_prediction[attack_rows].mean()), 'confusion_matrix': json.dumps(matrix.tolist())}
    for label in CLASSES:
        class_rows.append({'model': name, 'phase': phase, 'class': label, 'support': int(report[label]['support']), 'precision': report[label]['precision'], 'recall': report[label]['recall'], 'f1': report[label]['f1-score']})
        row[f'{label}_recall'] = report[label]['recall']
        row[f'{label}_f1'] = report[label]['f1-score']
    model_rows.append(row)
    models[name] = pipeline
    show_confusion(name, matrix)
    pd.DataFrame(model_rows).to_csv(RESULTS / 'model_comparison.csv', index=False)
    pd.DataFrame(class_rows).to_csv(RESULTS / 'per_class_metrics.csv', index=False)
    print(name, 'macro-F1:', round(row['macro_f1'], 4), 'normal FPR:', round(row['normal_false_positive_rate'], 4), 'seconds:', round(fit_seconds, 1))
    return row

## Initial model comparison

The tree models use sparse one-hot encoded data. HistGradientBoosting receives a dense matrix because that estimator expects dense input. XGBoost is included only when its package is already available.

In [ ]:
decision_tree = Pipeline([('preprocessor', make_preprocessor()), ('classifier', DecisionTreeClassifier(random_state=SEED))])
evaluate_model('Decision Tree', decision_tree, {'random_state': SEED})
random_forest = Pipeline([('preprocessor', make_preprocessor()), ('classifier', RandomForestClassifier(n_estimators=80, random_state=SEED, n_jobs=THREADS))])
evaluate_model('Random Forest', random_forest, {'n_estimators': 80, 'random_state': SEED, 'n_jobs': THREADS})
hist_gradient_boosting = Pipeline([('preprocessor', make_preprocessor(dense=True)), ('classifier', HistGradientBoostingClassifier(max_iter=150, random_state=SEED))])
evaluate_model('HistGradientBoosting', hist_gradient_boosting, {'max_iter': 150, 'random_state': SEED})
try:
    from xgboost import XGBClassifier
    xgb_model = Pipeline([('preprocessor', make_preprocessor()), ('classifier', XGBClassifier(objective='multi:softprob', num_class=4, n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=THREADS, random_state=SEED, eval_metric='mlogloss'))])
    label_codes = {label: number for number, label in enumerate(CLASSES)}
    xgb_start = time.perf_counter()
    xgb_model.fit(X_fit, y_fit.map(label_codes))
    xgb_seconds = time.perf_counter() - xgb_start
    xgb_prediction = np.array(CLASSES)[xgb_model.predict(X_validation).astype(int)]
    report = classification_report(y_validation, xgb_prediction, labels=CLASSES, output_dict=True, zero_division=0)
    matrix = confusion_matrix(y_validation, xgb_prediction, labels=CLASSES)
    normal_rows = y_validation.to_numpy() == 'Normal'
    attack_rows = ~normal_rows
    attack_prediction = xgb_prediction != 'Normal'
    xgb_row = {'model': 'XGBoost', 'phase': 'baseline', 'configuration': json.dumps({'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1, 'n_jobs': THREADS}), 'fit_seconds': xgb_seconds, 'accuracy': accuracy_score(y_validation, xgb_prediction), 'macro_f1': f1_score(y_validation, xgb_prediction, average='macro'), 'weighted_f1': f1_score(y_validation, xgb_prediction, average='weighted'), 'normal_false_positive_rate': float(attack_prediction[normal_rows].mean()), 'attack_recall': float(attack_prediction[attack_rows].mean()), 'confusion_matrix': json.dumps(matrix.tolist())}
    for label in CLASSES:
        class_rows.append({'model': 'XGBoost', 'phase': 'baseline', 'class': label, 'support': int(report[label]['support']), 'precision': report[label]['precision'], 'recall': report[label]['recall'], 'f1': report[label]['f1-score']})
        xgb_row[f'{label}_recall'] = report[label]['recall']
        xgb_row[f'{label}_f1'] = report[label]['f1-score']
    model_rows.append(xgb_row)
    models['XGBoost'] = xgb_model
    show_confusion('XGBoost', matrix)
except ImportError:
    print('XGBoost is not installed, so it was skipped.')
pd.DataFrame(model_rows).to_csv(RESULTS / 'model_comparison.csv', index=False)
pd.DataFrame(class_rows).to_csv(RESULTS / 'per_class_metrics.csv', index=False)
baseline_table = pd.DataFrame(model_rows).sort_values('macro_f1', ascending=False)
display(baseline_table[['model', 'macro_f1', 'accuracy', 'normal_false_positive_rate', 'attack_recall', 'fit_seconds']])

## Small grouped tuning experiment

The two highest baseline models are selected for this small search. Each candidate is scored with two grouped folds inside the four-class fit partition. The held-out validation partition is not used to choose parameters.

In [ ]:
top_two = baseline_table.head(2)['model'].tolist()
group_values = pd.util.hash_pandas_object(X_fit, index=False, categorize=True)
cv = list(StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED).split(X_fit, y_fit, group_values))
tuning_rows = []
tuned_rows = []
def grouped_cv(name, settings, make_model, use_weights=False):
    scores = []
    for train_part, check_part in cv:
        candidate = make_model(settings)
        weights = compute_sample_weight('balanced', y_fit.iloc[train_part]) if use_weights else None
        if weights is None:
            candidate.fit(X_fit.iloc[train_part], y_fit.iloc[train_part])
        else:
            candidate.fit(X_fit.iloc[train_part], y_fit.iloc[train_part], classifier__sample_weight=weights)
        scores.append(f1_score(y_fit.iloc[check_part], candidate.predict(X_fit.iloc[check_part]), average='macro'))
    tuning_rows.append({'model': name, 'configuration': json.dumps(settings, sort_keys=True), 'cv_macro_f1': float(np.mean(scores)), 'cv_std': float(np.std(scores)), 'folds': 2, 'weighted': use_weights})
    return tuning_rows[-1]

for model_name in top_two:
    if model_name == 'Decision Tree':
        candidates = [({'max_depth': None, 'min_samples_leaf': 1, 'class_weight': None}, False), ({'max_depth': 18, 'min_samples_leaf': 2, 'class_weight': 'balanced'}, False)]
        candidate_scores = []
        for settings, weighted in candidates:
            candidate_scores.append(grouped_cv(model_name, settings, lambda s: Pipeline([('preprocessor', make_preprocessor()), ('classifier', DecisionTreeClassifier(random_state=SEED, **s))]), weighted))
        best_settings = json.loads(max(candidate_scores, key=lambda row: row['cv_macro_f1'])['configuration'])
        tuned = Pipeline([('preprocessor', make_preprocessor()), ('classifier', DecisionTreeClassifier(random_state=SEED, **best_settings))])
        evaluate_model('Decision Tree tuned', tuned, best_settings, phase='tuned')
    elif model_name == 'Random Forest':
        candidates = [({'n_estimators': 50, 'max_depth': None, 'min_samples_leaf': 1, 'class_weight': None}, False), ({'n_estimators': 50, 'max_depth': 22, 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample'}, False)]
        candidate_scores = []
        for settings, weighted in candidates:
            candidate_scores.append(grouped_cv(model_name, settings, lambda s: Pipeline([('preprocessor', make_preprocessor()), ('classifier', RandomForestClassifier(random_state=SEED, n_jobs=THREADS, **s))]), weighted))
        best_settings = json.loads(max(candidate_scores, key=lambda row: row['cv_macro_f1'])['configuration'])
        tuned = Pipeline([('preprocessor', make_preprocessor()), ('classifier', RandomForestClassifier(random_state=SEED, n_jobs=THREADS, **best_settings))])
        tuned_rows.append(evaluate_model('Random Forest tuned', tuned, best_settings, phase='tuned'))
    elif model_name == 'HistGradientBoosting':
        candidates = [({'max_iter': 150, 'learning_rate': 0.08, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 0.1}, False), ({'max_iter': 200, 'learning_rate': 0.06, 'max_leaf_nodes': 15, 'min_samples_leaf': 30, 'l2_regularization': 0.5}, True)]
        candidate_scores = []
        for settings, weighted in candidates:
            candidate_scores.append(grouped_cv(model_name, settings, lambda s: Pipeline([('preprocessor', make_preprocessor(dense=True)), ('classifier', HistGradientBoostingClassifier(random_state=SEED, **s))]), weighted))
        best_row = max(candidate_scores, key=lambda row: row['cv_macro_f1'])
        best_settings = json.loads(best_row['configuration'])
        use_weights = best_row['weighted']
        tuned = Pipeline([('preprocessor', make_preprocessor(dense=True)), ('classifier', HistGradientBoostingClassifier(random_state=SEED, **best_settings))])
        weights = compute_sample_weight('balanced', y_fit) if use_weights else None
        tuned_rows.append(evaluate_model('HistGradientBoosting tuned', tuned, {**best_settings, 'sample_weight': 'balanced' if use_weights else None}, sample_weight=weights, phase='tuned'))
pd.DataFrame(tuning_rows).to_csv(RESULTS / 'tuning_log.csv', index=False)
pd.DataFrame(model_rows).to_csv(RESULTS / 'model_comparison.csv', index=False)
pd.DataFrame(class_rows).to_csv(RESULTS / 'per_class_metrics.csv', index=False)
print('Models selected for tuning:', top_two)
display(pd.DataFrame(tuning_rows))

## Four-class and six-class comparison

The four-class result and the six-class result describe different classification problems. Their macro-F1 values are shown together for context, but they are not directly comparable as a proof that one model is better at detecting every attack.

In [ ]:
comparison = pd.DataFrame(model_rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
eligible = comparison[comparison['normal_false_positive_rate'] <= 0.10]
best = eligible.iloc[0] if len(eligible) else comparison.iloc[0]
comparison['recommendation'] = 'REVIEW'
comparison.loc[comparison['model'] == best['model'], 'recommendation'] = 'KEEP'
comparison['reason'] = 'Compare macro-F1, minority recall, and normal false positives'
comparison.loc[comparison['model'] == best['model'], 'reason'] = 'Best four-class macro-F1 among candidates at or below 10% normal FPR' if len(eligible) else 'Highest four-class macro-F1'
comparison.to_csv(RESULTS / 'model_comparison.csv', index=False)
six_class = pd.read_csv(PROJECT / 'results' / 'stage3_finalist_comparison.csv')
six_class_best = six_class[six_class['selection'] == 'SELECTED'].iloc[0]
four_vs_six = pd.DataFrame([{'experiment': 'Six-class selected', 'model': six_class_best['model'], 'macro_f1': six_class_best['macro_f1'], 'accuracy': six_class_best['accuracy'], 'normal_false_positive_rate': six_class_best['normal_false_positive_rate'], 'minority_recall_summary': f"Analysis={six_class_best['Analysis_recall']:.3f}; Backdoor={six_class_best['Backdoor_recall']:.3f}; DoS={six_class_best['DoS_recall']:.3f}"}, {'experiment': 'Four-class selected', 'model': best['model'], 'macro_f1': best['macro_f1'], 'accuracy': best['accuracy'], 'normal_false_positive_rate': best['normal_false_positive_rate'], 'minority_recall_summary': f"Exploits={best['Exploits_recall']:.3f}; Fuzzers={best['Fuzzers_recall']:.3f}; DoS={best['DoS_recall']:.3f}"}])
four_vs_six.to_csv(RESULTS / 'four_vs_six_comparison.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 4))
comparison.set_index('model')[['macro_f1', 'accuracy', 'normal_false_positive_rate', 'attack_recall']].plot.bar(ax=ax, rot=25)
ax.set_ylim(0, 1)
ax.set_ylabel('Score or rate')
fig.tight_layout()
fig.savefig(PLOTS / 'model_comparison.png', dpi=140)
plt.close(fig)
best_model = models[best['model']]
joblib.dump({'pipeline': best_model, 'classes': CLASSES, 'predictors': predictors}, MODELS / 'best_four_class_model.joblib')
metadata = {'selected_model': best['model'], 'selection_reason': best['reason'], 'classes': CLASSES, 'predictors': predictors, 'fit_rows': len(fit_data), 'validation_rows': len(validation_data), 'macro_f1': float(best['macro_f1']), 'accuracy': float(best['accuracy']), 'normal_false_positive_rate': float(best['normal_false_positive_rate']), 'official_test_used': False}
(MODELS / 'best_four_class_model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
report = f"""# Four-class experiment summary

Selected model: {best['model']}

Validation macro-F1: {best['macro_f1']:.4f}

Validation accuracy: {best['accuracy']:.4f}

Normal false-positive rate: {best['normal_false_positive_rate']:.4f}

The four-class model keeps Normal, Exploits, Fuzzers, and DoS. Analysis and Backdoor are excluded from this experiment only. Four-class and six-class macro-F1 values describe different tasks and are not directly comparable.

The official test dataset was not loaded or evaluated.
"""
(RESULTS / 'experiment_summary.md').write_text(report, encoding='utf-8')
display(comparison[['model', 'macro_f1', 'accuracy', 'normal_false_positive_rate', 'attack_recall', 'recommendation', 'reason']])
display(four_vs_six)